# Verification of Stanza's Source Data's Accuracy

Stanza uses Universal Dependencies as a source of high quality, annotated datasets. However, the Latin datasets have been converted from a variety of standards. The PROIEL and Perseus treebanks both contain Caesar, so the training data provides Stanza with a couple different standards for the same text. 

My goal here is to evaluate, as best as possible, what the most common errors are likely to be in the training dataset. This will allow us to focus our data-cleaning processes on the most problematic cases in the dataset for the *Corpus Caesarianum* project.

The original Universal Dependencies data is available on GitHub. To clone both repositories (to whatever directory contains this repository), navigate there through your shell and enter these commands:

```bash
gh repo clone UniversalDependencies/UD_Latin-Perseus
gh repo clone UniversalDependencies/UD_Latin-PROIEL
```

These repositories contain not only the source data, but a document called **stats.xml** which has helpful statistics for each dataset. When more detail is required, we can use UD's official tools for comparing the datasets. In the same directory, call this command to clone the UD Tools repository:

```bash
gh repo clone UniversalDependencies/tools
```

Installing the Universal dependencies toolset requires an installation of Perl. Perl comes installed on most Linux and MacOS distributions, but Windows users need to [install Strawberry Perl](https://www.perl.org/get.html#win32). Once installed, use the `cpan` command to install the necessary dependencies.

```bash
cpan YAML JSON::Parse Lingua::Interset
#or, if cpanm is installed, install Lingua::Interset with that command instead
cpanm Lingua::Interset
```

The `lxml` module is required for dealing with XML files, and the `pyconll` module for the CONLLU treebank files. I install them on Python 3.14.5 here.

Installation commands:

```bash
pip install lxml pyconll
```

## Paths and Setup

We are dealing with all the CONLLU files in the repository, as the CONLLX format isn't available. Here is a full list of the files in each repository:

In [27]:
from pprint import pp
import os

paths_to_conllu = []
xml_paths = []

print(f"{'extension':>10} {'filename':>35} {'path':>50} {'is_file':>4} ")
print("-" * 94)
#note: casting DirEntry.path to string isn't necessary since the argument in os.scandir is a string, but it isn't guaranteed
for i in os.scandir("../UD_Latin-PROIEL"):
    if i.path.find(".conllu") > -1:
        paths_to_conllu.append(i.path) 
    if i.path.find(".xml") > -1:
        xml_paths.append(i.path)
    print(f"{i.name[i.name.rfind('.'):]:>10} {i.name:>35} {str(i.path):>50} {'yes' if i.is_file() else 'no':>4} ")
for i in os.scandir("../UD_Latin-Perseus"):
    if i.path.find(".conllu") > -1:
        paths_to_conllu.append(i.path) 
    if i.path.find(".xml") > -1:
        xml_paths.append(i.path)
    print(f"{i.name[i.name.rfind('.'):]:>10} {i.name:>35} {str(i.path):>50} {'yes' if i.is_file() else 'no':>4} ")

 extension                            filename                                               path is_file 
----------------------------------------------------------------------------------------------
      .git                                .git                            ../UD_Latin-PROIEL/.git   no 
.gitignore                          .gitignore                      ../UD_Latin-PROIEL/.gitignore  yes 
       .md                     CONTRIBUTING.md                 ../UD_Latin-PROIEL/CONTRIBUTING.md  yes 
      .txt                         LICENSE.txt                     ../UD_Latin-PROIEL/LICENSE.txt  yes 
       .md                           README.md                       ../UD_Latin-PROIEL/README.md  yes 
      .log                            eval.log                        ../UD_Latin-PROIEL/eval.log  yes 
   .conllu             la_proiel-ud-dev.conllu         ../UD_Latin-PROIEL/la_proiel-ud-dev.conllu  yes 
   .conllu            la_proiel-ud-test.conllu        ../UD_Latin-PROI

We saved all the CONLLU files to a list, `paths_to_conllu`, so we can work with them later.

In [28]:
print("CONLLU:")
pp(paths_to_conllu)
print("XML:")
pp(xml_paths)

CONLLU:
['../UD_Latin-PROIEL/la_proiel-ud-dev.conllu',
 '../UD_Latin-PROIEL/la_proiel-ud-test.conllu',
 '../UD_Latin-PROIEL/la_proiel-ud-train.conllu',
 '../UD_Latin-Perseus/la_perseus-ud-test.conllu',
 '../UD_Latin-Perseus/la_perseus-ud-train.conllu']
XML:
['../UD_Latin-PROIEL/stats.xml', '../UD_Latin-Perseus/stats.xml']


## Structure and Statistics of the Two Corpora

The structure of the *stats.xml* file is the same in each case, with a series of top-level elements for each type of tag, and sub-elements in each one showing the frequency of different values for that tag.

Here is an example structure:

```xml
<treebank>
<!-- Statistics of universal POS tags. The comments show the most frequent lemmas. -->
  <tags unique="14">
    <tag name="ADJ">11475</tag><!-- magnus, multus, bonus, publicus, sanctus, totus, primus, reliquus, alter, ceterus -->
    <tag name="ADP">16020</tag><!-- in, ad, ab, de, ex, cum, per, propter, super, pro -->
      <!-- ...-->
    <tag name="X">103</tag><!-- greek.expression -->
  </tags>
  <!-- Statistics of features and values. The comments show the most frequent word forms. -->
  <feats unique="44">
    <feat name="Aspect" value="Imp" upos="AUX,VERB">5023</feat><!-- erat, esset, erant, dicebant, essent, dicebat, posset, possent, habebat, habebant -->
    <feat name="Aspect" value="Perf" upos="AUX,VERB">13462</feat><!-- dixit, fuit, venit, factum, dixerunt, misit, facta, fecit, respondit, dedit -->
    <!-- ... -->
    <feat name="Voice" value="Pass" upos="AUX,VERB">9766</feat><!-- factum, fieri, facta, scriptum, factus, locutus, videtur, data, loqui, profectus -->
  </feats>
  <!-- Statistics of universal dependency relations. -->
  <deps unique="37">
    <dep name="acl">5373</dep>
    <!-- ... -->
    <dep name="xcomp">3467</dep>
  </deps>
</treebank>
```


Let's start by printing a side-by-side comparison of each.

In [77]:
from lxml import etree

def tostring_splitlines(root: Element, tag: str) -> list[str]:
    """
    Accepts a root element of stats.xml and a tag as arguments, and returns a list of strings including that element and all its subelements. 
    Each string in the list is a single line.
    """
    return etree.tostring(root.find(tag), pretty_print = True).decode().splitlines()

def print_tags_comparison(type: str):
    """
    Prints a side-by-side version of a certain type of element"""
    with open(xml_paths[0], 'r') as xml_1:
        with open(xml_paths[1], 'r') as xml_2:
            proiel = etree.parse(xml_1).getroot()
            perseus = etree.parse(xml_2).getroot()

            pers_tags = tostring_splitlines(perseus, type)
            proiel_tags = tostring_splitlines(proiel, type)
    
            lengths = [len(pers_tags), len(proiel_tags)]
            lengths.sort()
            max_lines = lengths[-1] #Get the maximum value
        
            print(f"{type}: {'Perseus':<180} {'PROIEL':<180}")
            print('-' * 150)
            for i in range(0, max_lines):
                try:
                    pers_val = pers_tags[i]
                except IndexError:
                    pers_val = ''
                try:
                    proiel_val = proiel_tags[i]
                except IndexError:
                    proiel_val = ''
                print(f"{pers_val:<180} {proiel_val:<180}")

Let's take a look at tags first. 

In [68]:
print_tags_comparison('tags')

tags: Perseus                                                                                                                            PROIEL                                                                                                                            
------------------------------------------------------------------------------------------------------------------------------------------------------
<tags unique="16">                                                                                                                 <tags unique="14">                                                                                                                
    <tag name="ADJ">2129</tag><!-- magnus, publicus, bonus, malus, novus, primus, parvus, superus, unus, gravis -->                    <tag name="ADJ">11475</tag><!-- magnus, multus, bonus, publicus, sanctus, totus, primus, reliquus, alter, ceterus -->         
    <tag name="ADP">1349</tag><!-- in, ad, ab, de, ex, cu

It's odd that particles (represented as PART in Perseus) don't exist in PROIEL, which only uses ADV for both. In the postagged data, *non* is always parsed as a PART, not an ADV, following Perseus practice, and same for *enim*.

PROIEL removes all punctuation beforehand, so the lack of a PUNCT tag is easy to understand.

Next, the dependency relations.

In [73]:
print_tags_comparison('deps')

deps: Perseus                                                                                                                            PROIEL                                                                                                                            
------------------------------------------------------------------------------------------------------------------------------------------------------
<deps unique="48">                                                                                                                 <deps unique="37">                                                                                                                
    <dep name="acl">86</dep>                                                                                                           <dep name="acl">5373</dep>                                                                                                    
    <dep name="acl:relcl">288</dep>                      

Now, finally, the UPOS features.


In [78]:
print_tags_comparison('feats')

feats: Perseus                                                                                                                                                                              PROIEL                                                                                                                                                                              
------------------------------------------------------------------------------------------------------------------------------------------------------
<feats unique="61">                                                                                                                                                                  <feats unique="44">                                                                                                                                                                 
    <feat name="AdpType" value="Post" upos="ADP">17</feat><!-- cum -->                                                

This last one bears more exploration. Perseus has numerous tags that are covered more concisely in the PROIEL dataset.

A more interesting question would be which 
